In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from manga_ocr import MangaOcr

# 1. Setup Paths
NOTEBOOK_DIR = os.getcwd()
# Adjust this depending on where your notebook is relative to the 'code' folder
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..')) 
sys.path.insert(0, os.path.join(BASE_DIR, 'code'))

# 2. Import Modules
from pipeline.Utils.MangaPipeline import MangaPipeline

# Helper function to display images in Jupyter
def show_img(img, figsize=(10, 10)):
    plt.figure(figsize=figsize, dpi=200)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
# 1. Initialize Pipeline (Loads models)
YOLO_PATH = os.path.join(BASE_DIR, 'best.pt')
pipeline = MangaPipeline(YOLO_PATH)

# 2. Define Image Path
IMG_PATH = os.path.join(BASE_DIR, "data/Manga109_released_2023_12_07/images/Belmondo/010.jpg")

# 3. Load Raw Image for processing
original_img_bgr = cv2.imread(IMG_PATH)
original_img_rgb = cv2.cvtColor(original_img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# IMAGE 1: Raw Input
show_img(original_img_rgb)

In [ ]:
# Run raw YOLO prediction directly
yolo_results = pipeline.segmenter.yolo_model.predict(source=IMG_PATH, verbose=False)
result = yolo_results[0]
raw_masks_points = result.masks.xy if result.masks else []

# Create visualization image
img_raw_detection = original_img_rgb.copy()
alpha = 0.2
overlay = img_raw_detection.copy()

for mask_points in raw_masks_points:
    # Convert points to integer format
    pts = np.array(mask_points, dtype=np.int32)
    
    # 1. Draw Mask (Filled with transparency or outline)
    cv2.fillPoly(overlay, [pts], color=(0, 255, 0))
cv2.addWeighted(overlay, alpha, img_raw_detection, 1-alpha, 0, img_raw_detection)
    
for mask_points in raw_masks_points:
    pts = np.array(mask_points, dtype=np.int32)
    # 2. Draw Bounding Box around the raw mask
    x, y, w, h = cv2.boundingRect(pts)
    cv2.rectangle(img_raw_detection, (x, y), (x+w, y+h), (0, 255, 0), 2)

# IMAGE 2: Raw Detection (Green)
show_img(img_raw_detection)

In [ ]:
# Run the segmenter
_, refined_bubbles = pipeline.segmenter.detect_and_segment(IMG_PATH)

# Create visualization image
img_refined_detection = original_img_rgb.copy()

for bubble in refined_bubbles:
    # Get mask and contour
    cnt = bubble['contour']
    x, y, w, h = bubble['bbox']
    
    # 1. Draw Refined Mask/Contour (Red)
    cv2.drawContours(img_refined_detection, [cnt], -1, (255, 0, 0), 2)
    
    # 2. Draw Refined Bounding Box
    cv2.rectangle(img_refined_detection, (x, y), (x+w, y+h), (255, 0, 0), 2)

# IMAGE 3: Refined Segmentation (Red)
show_img(img_refined_detection)

In [ ]:
import japanize_matplotlib
# Run OCR on the refined bubbles
img_ocr_viz = original_img_rgb.copy()

# Use Matplotlib to plot because OpenCV doesn't handle Japanese text well
plt.figure(figsize=(15, 20))
plt.imshow(img_ocr_viz)
plt.axis('off')

for bubble in refined_bubbles:
    x, y, w, h = bubble['bbox']
    
    # 1. Crop and Run OCR
    crop = original_img_rgb[y:y+h, x:x+w]
    ocr_text = pipeline.ocr_model.predict(crop)
    bubble['ocr_text'] = ocr_text # Save for later
    
    # 2. Draw Box (using plot to layer on top)
    rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='blue', facecolor='none')
    plt.gca().add_patch(rect)
    
    # 3. Write Text on top of BBox
    # Using a background box for readability
    plt.text(x, y - 5, ocr_text, fontsize=12, color='white', 
             bbox=dict(facecolor='blue', alpha=0.5))

plt.show()

In [ ]:
img_whitened = original_img_rgb.copy()
erosion_kernel = np.ones((6, 6), np.uint8)

for bubble in refined_bubbles:
    # Use original_mask (the raw shape before split) if available, else current mask
    # This ensures we cover the whole white area
    mask_to_use = bubble.get('original_mask', bubble['mask'])

    # Erode slightly so we don't erase the bubble border line
    eroded_mask = cv2.erode(mask_to_use, erosion_kernel, iterations=1)
    
    contours, _ = cv2.findContours(eroded_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Fill with White
    cv2.drawContours(img_whitened, contours, -1, (255, 255, 255), thickness=cv2.FILLED)

# IMAGE 5: Whitened Bubbles
show_img(img_whitened)

In [ ]:
# --- IMAGE 6: Final Translated Output ---
print("Translating...")
for bubble in refined_bubbles:
    ocr = bubble.get('ocr_text', '')
    
    if ocr.strip():
        # Run translation
        trans = pipeline.translator.predict(ocr)
        if isinstance(trans, list):
            trans = trans[0] if len(trans) > 0 else ""
        bubble['translated_text'] = str(trans)
    else:
        bubble['translated_text'] = ""

print("Typesetting...")
# 2. Typeset (Render)
final_image = pipeline.typesetter.render(original_img_rgb, refined_bubbles)

show_img(final_image)